<a href="https://colab.research.google.com/github/ROsorioL/tsins-model/blob/main/TSINS_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TSINS vs TIS — three-layer forearm model with an MRG axon

A verification model for **Transformed Synthesised Interferential Nerve Stimulation**
(Osorio, Edmondson, Sáez, Saavedra, Aqueveque, Poulton, Andrews & Jarvis), built to
compare TSINS against temporal interference stimulation (TIS) and a rectangular
biphasic control on three questions:

1. **action potential generation** — threshold and firing pattern of a myelinated motor axon,
2. **kilohertz blocking** — whether the antiphase blocking components really cancel at the
   nerve, and whether a single channel can block conduction,
3. **depth effect** — how all of this changes as the nerve is moved from 5 mm below the
   skin to the cylinder axis at 25 mm.

### What this model adds to the published one

| | published model | this model |
|---|---|---|
| volume conductor | single isotropic muscle cylinder, COMSOL | three layers (skin / fat / muscle) + nerve inclusion + electrode gel, finite volume, verified against a semi-analytic Bessel solution |
| axon | Hodgkin–Huxley | McIntyre–Richardson–Grill double cable, full myelin and periaxonal space |
| nerve position | on the cylinder axis (25 mm deep) | swept 5–25 mm; 5 mm is the ulnar nerve at the wrist |
| conductivity | single value at 8 kHz | separate Gabriel 4-Cole-Cole values for the harmonic band (20 Hz) and the kHz band |

### TSINS definition used here (paper Eqs. 1–6)

$$\mathrm{ch}_1(t)=g\Big[\sum_{n=n_0}^{n_0+K-1}A_n\sin(2\pi n f_0 t+\varphi_n)+A_b\sin(2\pi f_b t)\Big]$$
$$\mathrm{ch}_2(t)=g\Big[\sum_{n=n_0+K}^{n_0+2K-1}A_n\sin(2\pi n f_0 t+\varphi_n)+A_b\sin(2\pi f_b t+\pi)\Big]$$

with $f_0=2$ Hz, $n_0=4$, $K=7$, $A_n=1$, $\varphi_n=0$, $\kappa=10$ (so $A_b=10$) and $g=1/16$.
Because $\sin(\theta+\pi)=-\sin\theta$, the blocking components cancel wherever the two fields
add linearly, leaving $\mathrm{TSINS}(t)=\tfrac1{16}\sum_{n=4}^{17}\sin(2\pi n f_0 t)$.

**TIS** is defined here as $f_b$ on channel 1 and $f_b+2$ Hz on channel 2, so the beat
frequency matches the TSINS fundamental. **Biphasic control**: 0.25 ms per phase at 2 Hz,
delivered in phase on both channels (as in Fig. 8 of the paper).

### Running this notebook

**On Colab** the next cell clones the repository, installs `pyamg` (the one dependency Colab
does not ship) and switches into the repository directory, because the model reads its
`.npz` / `.csv` files from the working directory. It also enables Colab's custom widget
manager so the `ipywidgets` controls in sections 4 and 5.2 render.

**Locally** the cell does nothing - run the notebook from the repository root and the
imports in the following cell resolve against `./tsins_model/`.

A free CPU runtime is sufficient; there is no GPU code. The first numba-compiled simulation
costs roughly 20 s of JIT, after which one 0.5 s simulated period takes a few seconds.

In [ ]:
# --- Colab bootstrap (no-op outside Colab) --------------------------------
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ROsorioL/tsins-model.git"
REPO_DIR = "/content/tsins-model"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    # numpy, scipy, matplotlib, pandas, numba and ipywidgets are preinstalled on Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyamg"], check=True)
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()   # ipywidgets sliders in sections 4 and 5.2

print("python", sys.version.split()[0], "| colab:", IN_COLAB, "| working directory:", os.getcwd())
assert os.path.isdir("tsins_model") and os.path.isfile("leadfields.npz"), (
    "run this notebook from the repository root - tsins_model/ and leadfields.npz "
    f"were not found in {os.getcwd()}")

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import ipywidgets as W_ui
from IPython.display import display

from tsins_model import volume as V, tissue as T, mrg as M, waveforms as W, coupling as C, sweep as SW, blocktest as BT

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
FOC, ALT, GRN, GREY = "#1f6fb4", "#d1621a", "#2e7d32", "#8a8a8a"
LF = C.LeadFields("leadfields.npz")
print("lead fields available for nerve depths:", LF.available_depths(), "mm")

## 1  Tissue properties and the three-layer volume conductor

Conductivities come from the Gabriel 4-Cole-Cole dispersion model (IT'IS database).
Note the dispersion between the two bands that TSINS uses simultaneously.

In [ ]:
rows = []
for f in [2, 8, 20, 34, 3000, 8000, 10000]:
    rows.append(dict(f_Hz=f, muscle=T.cole_cole("muscle", f), fat=T.cole_cole("fat", f),
                     nerve=T.cole_cole("nerve", f), skin_dry=T.cole_cole("skin_dry", f)))
display(pd.DataFrame(rows).round(5))

print("\nSkin: the Gabriel dry-skin value (2.0e-4 S/m) implies ~150 kOhm through a 0.5 cm^2")
print("contact, which the DS5 (10 mA / 10 V range) cannot drive. The model therefore uses a")
print("prepared/wet skin value of 0.10 S/m by default, giving a channel resistance of ~1.1 kOhm.")
print("Set wet_skin=False in tissue.default_sigmas() to use the dry-skin dispersion instead.")
print("\nsigma set at 8 kHz:", {k: round(v, 5) for k, v in T.default_sigmas(8e3).items()})
print("sigma set at 20 Hz :", {k: round(v, 5) for k, v in T.default_sigmas(20.0).items()})

### Geometry

Cylinder of radius 25 mm and length 200 mm with insulating ends:

* muscle `r < 20.5 mm`, subcutaneous fat `20.5–23.5 mm`, skin `23.5–25 mm`
* nerve: a 3 mm diameter cylinder parallel to the axis, at depth `d` below the skin
* four cross-paired contacts, each two 8 mm pins 10 mm apart, at ±10 mm axially and
  ±10 mm circumferentially, with a 0.5 mm gel layer under each

`div(sigma grad phi) = 0` is discretised cell-centred on a graded `(r, theta, z)` grid and
solved with classical algebraic multigrid preconditioned CG.

In [ ]:
g = V.Geometry(nerve_depth=5.0)
print("contact centres (axial mm, arc mm):")
for k, v in V.contact_centres(g).items():
    print(f"   {k[0]}  {'anode ' if k[1] > 0 else 'cathode'}  z={v[0]:+5.1f}  s={v[1]:+5.1f}")
print(f"\nnerve axis at r = {g.r_nerve_axis} mm  (depth {g.nerve_depth} mm)")

z = LF.z
fig, axx = plt.subplots(1, 2, figsize=(9, 3.1))
for dep, c in zip([5, 10, 15, 20, 25], plt.cm.viridis(np.linspace(0, .85, 5))):
    axx[0].plot(z, LF.phi("lf", "ch1", dep), color=c, label=f"{dep} mm")
    d2 = np.gradient(np.gradient(LF.phi("lf", "ch1", dep), z), z)
    axx[1].plot(z, d2, color=c)
axx[0].set_xlim(-50, 50); axx[0].set_xlabel("axial position (mm)")
axx[0].set_ylabel(r"$\phi$ at the nerve (mV/mA)"); axx[0].legend(frameon=False, fontsize=7, title="depth")
axx[1].set_xlim(-50, 50); axx[1].set_xlabel("axial position (mm)")
axx[1].set_ylabel(r"$d^2\phi/dz^2$ (mV mm$^{-2}$ mA$^{-1}$)")
axx[1].set_title("activating function", fontsize=9)
fig.tight_layout(); plt.show()

**Verification of the volume conductor** (all three checks were run when the lead fields
were built; re-run the cell below to repeat them):

| check | result |
|---|---|
| finite volume vs semi-analytic Bessel solution (same 3 layers, no nerve/gel) | 0.45 % NRMSE, peak 48.9 vs 49.4 mV/mA |
| grid refinement, 320 k vs 1.07 M cells | 0.15 % NRMSE, 0.30 % peak difference |
| axial current crossing z = 0 for 1 mA injected | 1.0000 mA |
| channel resistance | 1.05–1.09 kΩ (consistent with the DS5 10 mA / 10 V compliance) |
| effect of including the nerve as a resistive inclusion | +36.7 % on $V_e$ at the axon (47.8 -> 65.3 mV/mA) |
| effect of including the gel pads | -2.9 % (67.3 -> 65.3 mV/mA) |

Set `RESOLVE = True` to rebuild a lead field from scratch (about 20 s per solve).

In [ ]:
RESOLVE = False
if RESOLVE:
    import tsins_model.spectral as S
    sig = T.default_sigmas(8e3)
    grid = V.make_grid(g, nr_muscle=40, nr_fat=7, nr_skin=6, nr_gel=3, ntheta=128, dz_fine=0.5)
    vc = V.VolumeConductor(g, sig, grid)
    x = vc.solve(vc.rhs("ch1", 1e-3))
    print(f"{vc.n_active} unknowns, {len(vc.last_residuals)} CG iterations, "
          f"R = {vc.electrode_impedance(x, 'ch1'):.0f} ohm")
    phi = vc.sample_line(x, g.r_nerve_axis, 0.0, np.linspace(-60, 60, 601))
    plt.plot(np.linspace(-60, 60, 601), phi * 1e3); plt.xlabel("z (mm)")
    plt.ylabel(r"$\phi$ (mV/mA)"); plt.show()
else:
    print("RESOLVE = False - using the stored lead fields in leadfields.npz")

## 2  The MRG axon

Full McIntyre–Richardson–Grill (2002) double cable: node, MYSA, FLUT and six STIN
compartments per internode, with the periaxonal space and myelin sheath as a second cable.
Backward Euler on the cable, exponential Euler on the gates, 2×2 block-tridiagonal solve
per step, numba-compiled and GIL-free.

The leak reversal is set so that the node is in exact steady state at −80 mV
(`balance_leak=True`); with `e_l = -90 mV` the node drifts.

In [ ]:
ax = M.MRGAxon(fiber_diam=10.0, n_nodes=51)
print(f"fibre 10 um: {ax.n} compartments, {ax.n_nodes} nodes, {ax.length_mm:.1f} mm long")
print(f"internode {ax.deltax} um, STIN length {ax.interlength:.2f} um, {ax.nl} myelin lamellae")
print(f"leak reversal {ax.e_l:.2f} mV; resting gates m,h,p,s = {np.round(M.steady_gates(), 4)}")
display(pd.read_csv("mrg_validation.csv"))

In [ ]:
# intracellular pulse at one end -> propagating action potential
dt = 0.002
tg = np.arange(0, 6 + dt, dt)
istim = np.where((tg >= 1.0) & (tg < 1.1), 3e-3, 0.0)
r = M.simulate(ax, np.zeros((ax.n, 1)), np.zeros((tg.size, 1)), dt,
               istim=istim, istim_idx=int(ax.node_index[2]), rec_dt=0.002)
zn = ax.node_positions()
fig, axx = plt.subplots(1, 2, figsize=(9, 3))
for j in range(0, len(zn), 6):
    axx[0].plot(r["t"], r["v"][:, j], lw=0.8, color=plt.cm.viridis(j / len(zn)))
axx[0].set_xlabel("time (ms)"); axx[0].set_ylabel(r"$V_m$ (mV)"); axx[0].set_xlim(1, 3.5)
axx[0].set_title(f"CV = {M.conduction_velocity(r['t'], r['v'], zn):.1f} m/s", fontsize=9)
j = len(zn) // 2
axx[1].plot(r["t"], r["v"][:, j], color=FOC)
axx[1].set_xlim(1.8, 12); axx[1].set_ylim(-85, 40)
axx[1].axhline(-80, color=GREY, lw=0.6, ls=":")
axx[1].set_xlabel("time (ms)"); axx[1].set_ylabel(r"$V_m$ (mV)")
axx[1].set_title("nodal AP with depolarising afterpotential", fontsize=9)
fig.tight_layout(); plt.show()

## 3  Waveform synthesis

Reproduction of the paper's Figure 1, and the delivered charge compared with the
measured values.

In [ ]:
fs = 2e6
t = np.arange(0, 1.0, 1 / fs)
d = W.tsins_channels(t, 8000.0)
fig, axx = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
axx[0].plot(t, d["ch1_lf"] + d["ch1_hf"], color=FOC, lw=0.3)
axx[0].plot(t, d["ch1_lf"], color="k", lw=1.0)
axx[0].set_ylabel("ch 1 (a.u.)"); axx[0].set_title("(a) harmonics 4-10 + 8 kHz blocker", fontsize=9, loc="left")
axx[1].plot(t, d["ch2_lf"] + d["ch2_hf"], color=ALT, lw=0.3)
axx[1].plot(t, d["ch2_lf"], color="k", lw=1.0)
axx[1].set_ylabel("ch 2 (a.u.)"); axx[1].set_title("(b) harmonics 11-17 + antiphase blocker", fontsize=9, loc="left")
axx[2].plot(t, d["target"], color="k", lw=1.0)
sq = np.sign(np.sin(2 * np.pi * 2 * t)) * 0
axx[2].set_ylabel("sum (a.u.)"); axx[2].set_xlabel("time (s)")
axx[2].set_title("(c) in-tissue sum: blockers cancel, target waveform remains", fontsize=9, loc="left")
fig.tight_layout(); plt.show()

rows = []
for wf, lab, meas in [("tsins", "TSINS", 6.96), ("tis", "TIS", 11.05), ("biphasic", "biphasic 2 Hz", 0.019)]:
    iv, _, dd = W.make_drive(wf, t, 8000.0, 10.0, rate=2.0)
    c1, c2, sm = W.channel_sum(iv)
    rows.append(dict(waveform=lab, ch_peak_mA=round(np.abs(c1).max(), 2),
                     sum_peak_mA=round(np.abs(sm).max(), 2),
                     charge_model_mC=round(W.delivered_charge(t, iv), 3), charge_measured_mC=meas))
df = pd.DataFrame(rows); df["model/measured"] = (df.charge_model_mC / df.charge_measured_mC).round(2)
display(df)
print(f"TSINS/TIS charge ratio: model {rows[0]['charge_model_mC']/rows[1]['charge_model_mC']:.3f}  |  paper 0.63")

## 4  Interactive explorer

Choose a waveform, blocking/carrier frequency, amplitude, nerve depth and fibre diameter,
then press **Run**. One 0.5 s period (one 2 Hz cycle) is simulated at dt = 2.5 µs; this takes
a few seconds. The window `[0.25, 0.75] s` is chosen because every waveform passes through
zero at both ends for integer-kHz carriers, so there is no switch-on transient.

In [ ]:
w_wave  = W_ui.Dropdown(options=[("TSINS", "tsins"), ("TIS", "tis"), ("biphasic control", "biphasic")],
                        value="tsins", description="waveform")
w_fb    = W_ui.FloatSlider(value=8.0, min=1.0, max=10.0, step=1.0, description="f_b (kHz)",
                           continuous_update=False)
w_amp   = W_ui.FloatSlider(value=2.0, min=0.25, max=20.0, step=0.25, description="amp (mA)",
                           continuous_update=False)
w_depth = W_ui.Dropdown(options=[5.0, 10.0, 15.0, 20.0, 25.0], value=5.0, description="depth (mm)")
w_fib   = W_ui.Dropdown(options=sorted(M.MRG_TABLE), value=10.0, description="fibre (um)")
w_disp  = W_ui.Dropdown(options=[("two-band (20 Hz + 8 kHz)", "two_band"), ("single (8 kHz)", "single")],
                        value="two_band", description="sigma(f)")
w_lead  = W_ui.Dropdown(options=[("aligned", "leadfields.npz"), ("offset 1 mm", "leadfields_offset1.npz"),
                                 ("offset 2 mm", "leadfields_offset2.npz"), ("offset 3 mm", "leadfields_offset3.npz"),
                                 ("offset 5 mm", "leadfields_offset5.npz")],
                        value="leadfields.npz", description="matrix")
w_go    = W_ui.Button(description="Run", button_style="primary")
w_out   = W_ui.Output()


def _run(_=None):
    with w_out:
        w_out.clear_output(wait=True)
        dep = 5.0 if w_lead.value != "leadfields.npz" else w_depth.value
        res = SW.run_one(w_wave.value, w_fb.value * 1e3, w_amp.value, depth=dep,
                         fiberD=w_fib.value, n_nodes=51, dispersion=w_disp.value,
                         lead_path=w_lead.value, return_traces=True)
        tr = res["traces"]
        ve = tr["ve"]; jm = int(np.argmax(np.abs(ve).max(axis=0)))
        tt = (tr["t_drive"] - tr["t_drive"][0]) * 1e3
        fig, axx = plt.subplots(2, 2, figsize=(10, 5.4))
        c1 = tr["ivec"][:, 0] + tr["ivec"][:, 1]; c2 = tr["ivec"][:, 2] + tr["ivec"][:, 3]
        axx[0, 0].plot(tt, c1, color=FOC, lw=0.3); axx[0, 0].plot(tt, c2, color=ALT, lw=0.3)
        axx[0, 0].plot(tt, c1 + c2, color="k", lw=0.6)
        axx[0, 0].set_ylabel("current (mA)"); axx[0, 0].set_title("channel currents and sum", fontsize=9)
        axx[0, 1].plot(tt, ve[:, jm], color=FOC, lw=0.5)
        axx[0, 1].set_ylabel(r"$V_e$ (mV)"); axx[0, 1].set_title("extracellular potential at the axon", fontsize=9)
        sig = ve[:, jm] - ve[:, jm].mean()
        dts = tr["t_drive"][1] - tr["t_drive"][0]
        S = np.abs(np.fft.rfft(sig)) * 2 / sig.size; f = np.fft.rfftfreq(sig.size, dts)
        axx[1, 0].semilogy(f, np.maximum(S, 1e-15), color=FOC, lw=0.5)
        axx[1, 0].set_xlim(0, 12000); axx[1, 0].set_ylim(1e-14, 1e3)
        axx[1, 0].axvline(w_fb.value * 1e3, color=ALT, lw=0.8, ls="--")
        axx[1, 0].set_xlabel("frequency (Hz)"); axx[1, 0].set_ylabel(r"$|V_e|$ (mV)")
        axx[1, 0].set_title(f"spectrum  ($f_b$ residual "
                            f"{S[np.argmin(np.abs(f - w_fb.value*1e3))]:.2e} mV)", fontsize=9)
        v = tr["v"]
        im = axx[1, 1].imshow(v.T, aspect="auto", origin="lower", cmap="magma", vmin=-90, vmax=30,
                              extent=[tr["t"][0], tr["t"][-1], tr["z"][0], tr["z"][-1]])
        axx[1, 1].set_xlabel("time (ms)"); axx[1, 1].set_ylabel("position (mm)")
        axx[1, 1].set_title(r"$V_m$ along the axon", fontsize=9)
        fig.colorbar(im, ax=axx[1, 1], pad=0.02, label="mV")
        axx[0, 0].set_xlabel("time (ms)"); axx[0, 1].set_xlabel("time (ms)")
        fig.tight_layout(); plt.show()
        print(f"propagated APs: {res['n_ap_propagated']}  ({res['ap_rate_hz']:.0f} per second)")
        print(f"APs at the most active node: {res['n_ap_max']};  "
              f"fraction of nodes firing: {res['frac_nodes_firing']:.2f}")
        print(f"initiation site: z = {res['init_z_mm']:+.1f} mm;  peak |Ve| = {res['ve_peak_mV']:.1f} mV")
        print(f"delivered charge over the period: {res['charge_mC']:.4f} mC")


w_go.on_click(_run)
display(W_ui.VBox([W_ui.HBox([w_wave, w_fb, w_amp]),
                   W_ui.HBox([w_depth, w_fib, w_disp]),
                   W_ui.HBox([w_lead, w_go]), w_out]))
_run()

### Threshold finder

`sweep.threshold` runs an ascending geometric scan followed by a bracketed bisection.
A plain bisection is wrong here: kHz stimulation is **not monotone** in amplitude — an axon
that fires at threshold can be conduction-blocked at higher amplitudes.

In [ ]:
w_twave = W_ui.Dropdown(options=[("TSINS", "tsins"), ("TIS", "tis"), ("biphasic", "biphasic")],
                        value="tsins", description="waveform")
w_tfb = W_ui.FloatSlider(value=8.0, min=1.0, max=10.0, step=1.0, description="f_b (kHz)", continuous_update=False)
w_tdep = W_ui.Dropdown(options=[5.0, 10.0, 15.0, 20.0, 25.0], value=5.0, description="depth (mm)")
w_tgo = W_ui.Button(description="Find threshold", button_style="primary")
w_tout = W_ui.Output()

def _thr(_=None):
    with w_tout:
        w_tout.clear_output(wait=True)
        print("searching ...")
        th = SW.threshold(w_twave.value, w_tfb.value * 1e3, depth=w_tdep.value, lo=0.2, hi=40.0)
        w_tout.clear_output(wait=True)
        print(f"{w_twave.value} at f_b = {w_tfb.value:.0f} kHz, nerve {w_tdep.value:.0f} mm deep:")
        print(f"    threshold = {th:.3f} mA per channel" if np.isfinite(th)
              else "    no propagated AP up to 40 mA")

w_tgo.on_click(_thr)
display(W_ui.VBox([W_ui.HBox([w_twave, w_tfb, w_tdep, w_tgo]), w_tout]))

## 5  Results

### 5.1  Action potential generation and carrier dependence

Nerve 5 mm below the skin, 10 µm fibre, one 2 Hz period per condition.

In [ ]:
thr = pd.read_csv("sweep_thresholds.csv"); run = pd.read_csv("sweep_runs.csv")
t5 = thr[thr.depth == 5].pivot_table(index="fb", columns="waveform", values="threshold_mA")
display(t5.round(3))
fig, axx = plt.subplots(1, 2, figsize=(9.5, 3.2))
for wf, c, lab in [("tsins", FOC, "TSINS"), ("tis", ALT, "TIS")]:
    s = thr[(thr.waveform == wf) & (thr.depth == 5)].sort_values("fb")
    axx[0].plot(s.fb / 1e3, s.threshold_mA, "o-", color=c, label=lab, ms=4)
bt = float(thr[(thr.waveform == "biphasic") & (thr.depth == 5)].threshold_mA.iloc[0])
axx[0].axhline(bt, color=GREY, ls="--", lw=1); axx[0].text(9.8, bt + .2, "biphasic 2 Hz",
                                                           fontsize=7, color=GREY, ha="right")
axx[0].set_xlabel("blocking / carrier frequency (kHz)"); axx[0].set_ylabel("threshold (mA)")
axx[0].legend(frameon=False); axx[0].set_ylim(0, 8)
for fb in sorted(run[run.waveform == "tis"].fb.unique()):
    for wf, a in [("tsins", axx[1]), ("tis", axx[1])]:
        s = run[(run.waveform == wf) & (run.fb == fb) & (run.depth == 5)].sort_values("amp_mA")
        a.plot(s.amp_mA, s.ap_rate_hz, "-", lw=0.9,
               color=FOC if wf == "tsins" else ALT, alpha=0.8)
axx[1].set_xlabel("channel amplitude (mA)"); axx[1].set_ylabel(r"propagated APs (s$^{-1}$)")
axx[1].text(0.03, 0.95, "TSINS (blue): eight superimposed curves\nTIS (orange): one per carrier",
            transform=axx[1].transAxes, va="top", fontsize=7)
fig.tight_layout(); plt.show()

**Result.** TSINS threshold is 0.83 mA at *every* blocking frequency from 3 to 10 kHz;
TIS rises from 3.03 mA at 3 kHz to 7.18 mA at 10 kHz (2.4-fold). This is the
waveform × frequency interaction the paper reports, obtained here from first principles.

In this symmetric layout the two channels reach the nerve with identical weight (agreement
to 1e-6 mV/mA), so the blocking components cancel **exactly** and the carrier independence
of TSINS is exact by construction. Section 5.2 tests how much channel imbalance it takes to
break it.

### 5.2  Kilohertz blocking

In [ ]:
blk = pd.read_csv("block_thresholds.csv")
blk["blocker_current_mA"] = blk.block_threshold_mA * W.G_DEFAULT * W.KAPPA_DEFAULT
display(blk.round(3))
cv = pd.read_csv("cancellation_vs_offset.csv"); display(cv.round(3))
print("\n'blocker' = both channels driven antiphase (the in-tissue sum): NaN means no block")
print("was produced at any amplitude tested, i.e. the cancelling pair cannot block conduction.")

In [ ]:
# live block test: a train of 5 test APs is launched proximally and counted distally
w_car = W_ui.Dropdown(options=[("one channel alone", "blocker_1ch"),
                               ("antiphase pair (in-tissue sum)", "blocker"),
                               ("TSINS channel 1 (harmonics + blocker)", "ch1_only"),
                               ("no carrier", "none")], value="blocker_1ch", description="carrier")
w_bfb = W_ui.FloatSlider(value=8.0, min=3.0, max=10.0, step=1.0, description="f_b (kHz)", continuous_update=False)
w_bamp = W_ui.FloatSlider(value=40.0, min=2.0, max=120.0, step=2.0, description="amp (mA)", continuous_update=False)
w_bgo = W_ui.Button(description="Run block test", button_style="primary"); w_bout = W_ui.Output()

def _blk(_=None):
    with w_bout:
        w_bout.clear_output(wait=True)
        r = BT.block_run(w_car.value, w_bfb.value * 1e3, w_bamp.value, return_traces=True)
        tr = r["traces"]
        fig, a = plt.subplots(figsize=(8, 2.8))
        a.plot(tr["t"], tr["v"][:, 3], color=GREY, lw=0.6, label="proximal node (stimulated)")
        a.plot(tr["t"], tr["v"][:, -4] - 140, color=FOC, lw=0.6, label="distal node (detector)")
        a.set_xlabel("time (ms)"); a.set_yticks([]); a.legend(frameon=False, fontsize=7, loc="upper left")
        a.set_title(f"{r['n_arrived']}/{r['n_test']} test APs arrived "
                    f"({'BLOCKED' if r['blocked'] else 'conducting'})", fontsize=9)
        fig.tight_layout(); plt.show()
        print(f"peak |Ve| at the axon: {r['ve_peak_mV']:.1f} mV; onset APs during settling: {r['n_onset_aps']}")

w_bgo.on_click(_blk)
display(W_ui.VBox([W_ui.HBox([w_car, w_bfb, w_bamp, w_bgo]), w_bout]))

**Result.**

* A **single channel's** kilohertz component blocks conduction at 8.9 mA (3 kHz) rising to
  17.4 mA (10 kHz) of blocker current — the usual increase of kHz block threshold with
  frequency. This is the superficial-tissue situation, under one electrode pair, and is the
  mechanism the paper proposes for reduced cutaneous recruitment.
* The **antiphase pair** produced no block at any amplitude tested (up to 94 mA of blocker
  current), because the two components cancel where the fields overlap.
* Misaligning the electrode matrix by 1–5 mm leaves 21–62 % of one channel's field at $f_b$
  at the nerve, i.e. 0.27–0.69 times the activating function of the reconstructed harmonic
  waveform. Even so, the TSINS threshold only moves from 0.83 to 0.85 mA and stays flat
  across 3–10 kHz: the residual carrier is low-pass filtered by the axon membrane and does
  not contribute to firing at these amplitudes. **TSINS does not require perfect
  cancellation** — consistent with the paper's Discussion.

### 5.3  Depth effect

In [ ]:
tbl = thr.pivot_table(index="depth", columns=["waveform", "fb"], values="threshold_mA")
display(tbl.round(2))
fig, a = plt.subplots(figsize=(5.2, 3.4))
for wf, fb, c, ls, lab in [("tsins", 3000., FOC, "-", "TSINS 3 kHz"), ("tsins", 8000., FOC, "--", "TSINS 8 kHz"),
                           ("tis", 3000., ALT, "-", "TIS 3 kHz"), ("tis", 8000., ALT, "--", "TIS 8 kHz")]:
    s = thr[(thr.waveform == wf) & (thr.fb == fb)].sort_values("depth")
    a.plot(s.depth, s.threshold_mA, ls, marker="o", color=c, ms=3.5, label=lab)
sb = thr[thr.waveform == "biphasic"].sort_values("depth")
a.plot(sb.depth, sb.threshold_mA, "-o", color=GREY, ms=3.5, label="biphasic 2 Hz")
a.axhline(10, color="k", lw=0.6, ls=":"); a.text(24.5, 10.6, "10 mA protocol limit", fontsize=7, ha="right")
a.set_xlabel("nerve depth (mm)"); a.set_ylabel("threshold (mA)"); a.set_ylim(0, 40)
a.legend(frameon=False, fontsize=7); fig.tight_layout(); plt.show()

**Result.** Thresholds rise with depth for every waveform, but at very different rates.
Over 5 → 25 mm the TSINS threshold rises 9.1-fold (0.83 → 7.51 mA) and the biphasic control
14-fold (1.00 → 14.0 mA), while TIS at 3 kHz rises 12-fold by 20 mm (3.03 → 36.7 mA) and is
already beyond 40 mA at 25 mm; TIS at 8 kHz exceeds 40 mA beyond 10 mm.

Two consequences:

1. **TSINS is the only one of the three that still activates the axon within the 10 mA
   protocol range at every depth tested.** Beyond ~12 mm it also beats the biphasic control.
2. The published on-axis geometry (nerve at 25 mm) is a much harder case than the real ulnar
   nerve at the wrist. The peak potential only falls 2.8-fold from 5 to 25 mm, but the peak
   **activating function** falls 21-fold, because on the axis the potential profile along the
   nerve is nearly linear between the two end plateaux. This is why the published model needs
   a far larger current to fire a mammalian myelinated axon than the experiment used.

## 6  Verification summary and limitations

**Verified against the paper**

| quantity | this model | paper |
|---|---|---|
| blocking component at the nerve (aligned electrodes) | 2 × 10⁻¹¹ mV | "no 8 kHz component was visible" |
| peak $|V_e|$ at the on-axis nerve, TSINS | 41 mV at 1.69 mA (symmetric, so +41 / −41) | +41 / −32 mV |
| main lobe of the target waveform | t = 0.4897 s | "main lobe at t = 0.5 s" |
| charge per 1 s at 10 mA, TSINS / TIS | 0.636 | 0.63 |
| charge, TSINS / TIS / biphasic | 8.10 / 12.73 / 0.0200 mC | 6.96 / 11.05 / 0.019 mC |
| TSINS carrier dependence of recruitment | none (0.83 mA at 3–10 kHz) | "near-identical recruitment curves" |
| TIS carrier dependence | threshold ×2.4 from 3 to 10 kHz | force falls steeply with carrier |

**Verified independently**

| component | check |
|---|---|
| volume conductor | 0.45 % NRMSE vs a semi-analytic Bessel solution; grid-converged to 0.15 %; exact current conservation |
| MRG axon | CV 50.5 m/s at 10 µm (MRG 2002 ≈ 52); CV/D 4.0–5.3; AP 110 mV, 0.33 ms half-width, DAP present; chronaxie 148 µs, rheobase 48 µA |
| time step | 2.5 µs converged to < 1 % against 0.5 µs for a 10 kHz carrier |

**Limitations**

1. **One fibre, not a population.** Force in the experiment reflects how many motor axons are
   recruited. This model simulates a single 10 µm fibre at one position, so it predicts
   thresholds and firing patterns, not force. At 10 mA the model still fires for TIS at
   10 kHz (above its 7.18 mA threshold) even though the experiment found almost no force
   there — a population model with a distribution of diameters and distances is needed to
   reproduce that collapse.
2. **Exact symmetry.** With the aligned cross-paired layout the two channels are
   electrically indistinguishable at the nerve, so blocking cancellation is exact. The real
   asymmetry reported in the paper's Discussion is reproduced only by offsetting the matrix.
3. **Skin conductivity** is set to a prepared/wet value of 0.10 S/m rather than the Gabriel
   dry-skin value, which is incompatible with the DS5 compliance voltage. Results scale with
   this choice; it is a parameter in `tissue.default_sigmas`.
4. **Quasi-static, linear, isotropic tissue.** No muscle anisotropy, no electrode–skin
   interface impedance, no temperature or accommodation effects, no cutaneous afferents (so
   the model says nothing directly about discomfort).
5. The nerve is a straight cylinder of uniform conductivity; fascicular structure and
   perineurium are not represented.